# Análise Bivariada e Multivariada

## Introdução

No campo da ciência de dados, a jornada para extrair insights começa frequentemente com a compreensão de variáveis individuais. A análise univariada, o estudo de uma única variável por vez, permite-nos resumir e descrever as suas características centrais, como a média de idade de um cliente ou a distribuição dos preços das casas.

Embora fundamental, esta abordagem apenas arranha a superfície. O mundo real é um sistema complexo de variáveis interconectadas; o preço de uma casa não depende apenas da sua área, mas também da localização, do número de quartos e da taxa de criminalidade do bairro. Para compreender verdadeiramente estes sistemas, devemos ir além das variáveis isoladas e explorar as suas relações.

Esta aula é dedicada a essa transição crucial: da análise de uma variável para a análise de duas (bivariada) e, finalmente, para a análise de muitas (multivariada). O objetivo da análise bivariada e multivariada é descobrir, quantificar e visualizar as relações e estruturas complexas que se escondem nos dados.

Procuraremos responder a perguntas orientadoras que estão no cerne da análise de dados do mundo real:

* Um maior investimento em publicidade leva a mais vendas? (Relação bivariada)
* Que características definem os nossos segmentos de clientes mais lucrativos? (Estrutura multivariada)
* Podemos simplificar um conjunto de dados complexo com dezenas de características sem perder informações cruciais? (Redução de dimensionalidade)

Um analista procura primeiro compreender os componentes individuais de um sistema (análise univariada), depois as suas relações diretas (análise bivariada) e, finalmente, a interação complexa de todo o sistema (análise multivariada).

### Configuração do Ambiente

Antes de começarmos a nossa exploração, vamos preparar o nosso ambiente de trabalho. A célula de código abaixo importa todas as bibliotecas Python necessárias que utilizaremos ao longo desta aula.

In [ ]:
# Importação de bibliotecas essenciais para manipulação de dados e computação numérica
import numpy as np
import pandas as pd

# Importação de bibliotecas de visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Importação de módulos do scikit-learn para pré-processamento, redução de dimensionalidade e clustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.cluster import KMeans
from sklearn.datasets import load_wine

# Configurações para melhorar a estética das visualizações
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

print("Bibliotecas importadas com sucesso!")

## Parte 1: Análise Bivariada – A Arte de Ver Relações

A análise bivariada é o pilar da descoberta de relações. Foca-se na análise de variáveis em pares para determinar se existe uma associação entre elas e, em caso afirmativo, qual a sua natureza e força. Iremos explorar sistematicamente as três combinações possíveis de tipos de variáveis: numérico-numérico, categórico-categórico e numérico-categórico.

### Análise de Pares de Variáveis Numéricas (Numérico-Numérico)

Quando lidamos com duas variáveis numéricas, como a altura e o peso de uma pessoa, o nosso principal objetivo é quantificar como elas se movem em conjunto.

#### A Matemática das Relações: Covariância vs. Correlação

Para entender a relação entre duas variáveis numéricas, começamos com dois conceitos estatísticos fundamentais: covariância e correlação.

A **covariância** é uma medida da variabilidade conjunta de duas variáveis. Indica a direção da relação linear entre elas. A fórmula para a covariância da amostra entre duas variáveis $X$ e $Y$ é:

$$Cov(X, Y) = \frac{\sum_{i=1}^{n}(X_i - \bar{X})(Y_i - \bar{Y})}{n-1}$$

* Se $Cov(X, Y) > 0$, as variáveis tendem a mover-se na mesma direção (relação positiva).
* Se $Cov(X, Y) < 0$, as variáveis tendem a mover-se em direções opostas (relação negativa).
* Se $Cov(X, Y) \approx 0$, não há uma relação linear clara.

No entanto, a covariância tem uma limitação significativa: o seu valor é sensível à escala das variáveis. Por exemplo, a covariância entre altura (em metros) e peso (em quilogramas) será muito diferente da covariância entre altura (em centímetros) e peso (em gramas), mesmo que a relação subjacente seja idêntica. Isto torna a magnitude da covariância difícil de interpretar e comparar entre diferentes conjuntos de dados.

Esta limitação motiva a introdução do **Coeficiente de Correlação de Pearson**, que é uma versão padronizada da covariância. Ao dividir a covariância pelo produto dos desvios padrão das duas variáveis, a correlação torna-se uma métrica sem unidades, limitada entre -1 e 1. Isto permite-lhe medir não só a *direção*, mas também a *força* da relação linear. A fórmula é:

$$\rho(X, Y) = \frac{Cov(X, Y)}{\sigma_X \sigma_Y}$$

O valor de $\rho$ (rho) é interpretado da seguinte forma:
* $\rho = 1$: Correlação linear positiva perfeita.
* $\rho = -1$: Correlação linear negativa perfeita.
* $\rho = 0$: Nenhuma correlação linear.
* Valores próximos de 1 ou -1 indicam uma forte relação linear, enquanto valores próximos de 0 indicam uma relação linear fraca ou inexistente.

A tabela seguinte resume as principais diferenças entre estes dois conceitos, servindo como uma referência clara para solidificar a sua compreensão.

| Característica | Covariância | Correlação |
| :--- | :--- | :--- |
| **Definição** | Mede a variabilidade conjunta de duas variáveis. | Mede a força e a direção de uma relação linear. |
| **Valores** | Ilimitados ($-\infty$ a $+\infty$). | Limitados (-1 a +1). |
| **Unidades** | Produto das unidades das duas variáveis. | Sem unidades. |
| **Dependência da Escala** | Altamente dependente da escala das variáveis. | Independente da escala das variáveis. |
| **Interpretação** | O sinal indica a direção (+/-); a magnitude não é comparável. | O sinal indica a direção; a magnitude indica a força. |

#### Implementação em Python

Vamos aplicar estes conceitos utilizando o conjunto de dados `tips` da biblioteca Seaborn, que contém informações sobre gorjetas dadas em um restaurante.

In [ ]:
# Carregar o conjunto de dados 'tips'
tips_df = sns.load_dataset('tips')

# Exibir as primeiras linhas do dataframe
print("Amostra do conjunto de dados 'tips':")
print(tips_df.head())

print(tips_df.describe())


In [ ]:

# Selecionar apenas as colunas numéricas para o cálculo
numerical_cols = tips_df.select_dtypes(include=np.number)

# Calcular a matriz de covariância usando pandas
print("\nMatriz de Covariância:")
print(numerical_cols.cov())

# Calcular a matriz de correlação usando pandas
print("\nMatriz de Correlação:")
print(numerical_cols.corr())

A matriz de correlação mostra-nos que existe uma correlação positiva de aproximadamente 0.676 entre `total_bill` (conta total) e `tip` (gorjeta). Este é um valor moderadamente forte, sugerindo que, como esperado, contas mais altas tendem a resultar em gorjetas mais altas.

#### Visualização: Dominando o Gráfico de Dispersão (Scatter Plot)

Embora um número como o coeficiente de correlação seja útil, ele resume uma relação complexa num único valor. A visualização é essencial para uma compreensão mais profunda. O gráfico de dispersão (scatter plot) é a principal ferramenta visual para analisar a relação entre duas variáveis numéricas. Cada ponto no gráfico representa uma observação, com as suas coordenadas x e y correspondendo aos valores das duas variáveis.

Um gráfico de dispersão não é apenas uma forma de apresentar resultados; é um instrumento primário de investigação. Ele revela padrões — linearidade, força, outliers e clusters — que um único número de correlação não consegue capturar. Um coeficiente de correlação de 0.7 é apenas um número, mas um gráfico de dispersão pode revelar que este valor é impulsionado por alguns outliers influentes, ou que a relação é, na verdade, não linear (por exemplo, curva), algo que a correlação de Pearson não captaria bem. Portanto, um fluxo de trabalho analítico robusto não se limita a calcular `df.corr()`, mas segue imediatamente com `sns.scatterplot()`. É fundamental incutir este hábito: visualizar sempre a relação que se está a medir.

In [ ]:
# Criar um gráfico de dispersão para visualizar a relação entre 'total_bill' e 'tip'
plt.figure(figsize=(12, 7))
sns.scatterplot(data=tips_df, x='total_bill', y='tip')
plt.title('Relação entre o Valor da Conta e a Gorjeta')
plt.xlabel('Valor Total da Conta ($)')
plt.ylabel('Valor da Gorjeta ($)')
plt.show()

O gráfico confirma visualmente a relação linear positiva. Podemos enriquecer ainda mais esta visualização adicionando uma terceira variável (categórica) através do parâmetro `hue`. Isto permite-nos ver se a relação muda entre diferentes grupos.

In [ ]:
# Adicionar uma terceira dimensão (smoker) usando a cor (hue)
plt.figure(figsize=(12, 7))
sns.scatterplot(data=tips_df, x='total_bill', y='tip', hue='smoker', style='time')
plt.title('Relação Conta vs. Gorjeta, distinguindo Fumadores e Não-Fumadores')
plt.xlabel('Valor Total da Conta ($)')
plt.ylabel('Valor da Gorjeta ($)')
plt.legend()
plt.show()

#### Exercício 1: Explorando o Conjunto de Dados `Tips`

**Tarefa:** Calcule o coeficiente de correlação entre `total_bill` e `tip`. Crie um gráfico de dispersão para estas duas variáveis, colorindo os pontos pelo `sex`. Interprete os resultados: A relação é forte? É positiva ou negativa? A relação parece mudar entre os sexos?

### Análise de Pares de Variáveis Categóricas (Categórico-Categórico)

Quando analisamos duas variáveis categóricas, como o dia da semana e se o cliente era fumador, o nosso interesse reside em frequências e proporções. Queremos saber se a ocorrência de uma categoria numa variável está associada à ocorrência de uma categoria na outra.

#### Fundamentos: Tabelas de Contingência e o Teste Qui-Quadrado

A ferramenta principal para esta análise é a **tabela de contingência** (ou tabulação cruzada), que mostra a distribuição de frequência de uma variável em cada categoria da outra.

Para determinar se a associação observada é estatisticamente significativa, utiliza-se o **Teste Qui-Quadrado ($ \chi^2 $) de Independência**. Este teste compara as frequências observadas na tabela de contingência com as frequências que seriam esperadas se não houvesse associação entre as variáveis. Um valor de p baixo (tipicamente < 0.05) sugere que a associação observada não se deve ao acaso.

#### Implementação e Visualização

Podemos criar facilmente uma tabela de contingência com a função `crosstab` do Pandas. No entanto, uma tabela de números pode ser difícil de interpretar rapidamente. Uma visualização, como um mapa de calor (heatmap), transforma a tabela numa grelha codificada por cores, tornando os padrões imediatamente aparentes.

In [ ]:
# Criar uma tabela de contingência entre 'day' e 'smoker'
contingency_table = pd.crosstab(tips_df['day'], tips_df['smoker'])
print("Tabela de Contingência (Dia vs. Fumador):")
print(contingency_table)

# Visualizar a tabela de contingência com um heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(contingency_table, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Contagem de Clientes por Dia da Semana e Status de Fumador')
plt.xlabel('Fumador')
plt.ylabel('Dia da Semana')
plt.show()

O mapa de calor torna claro que o sábado é o dia com o maior número de clientes fumadores, enquanto o domingo tem o maior número de não-fumadores.

#### Exercício 2: Género e Dia da Semana

**Tarefa:** Crie uma tabela de contingência para as variáveis `sex` e `day` no conjunto de dados `tips`. Visualize esta tabela com um mapa de calor. Com base na visualização, em que dia é que o restaurante parece ter a maior proporção de clientes do sexo feminino a pagar a conta?

### Análise de Pares Numéricos e Categóricos (Numérico-Categórico)

Neste cenário, o objetivo é comparar a distribuição de uma variável numérica entre os diferentes níveis de uma variável categórica. Por exemplo: "Como é que a distribuição dos valores das gorjetas difere entre fumadores e não-fumadores?"

#### A Visualização como Análise

Aqui, a visualização não é apenas um complemento, é a própria análise. A biblioteca Seaborn oferece um conjunto de gráficos ideais para esta tarefa, cada um oferecendo uma perspetiva diferente:

* **Box Plot:** Excelente para comparar medianas, quartis e identificar outliers entre grupos. Mostra um resumo de cinco números da distribuição.
* **Violin Plot:** Combina um box plot com uma estimativa de densidade de kernel (KDE). Mostra a distribuição de probabilidade completa dos dados, revelando picos e a forma da distribuição de forma mais detalhada.


Vamos usar estes gráficos para comparar o `total_bill` nos diferentes dias da semana.

In [ ]:
# Definir a ordem dos dias da semana para uma melhor visualização
day_order = ['Thur', 'Fri', 'Sat', 'Sun']

# Criar um box plot para comparar 'total_bill' por 'day'
plt.figure(figsize=(12, 7))
sns.boxplot(data=tips_df, x='day', y='total_bill', order=day_order)
plt.title('Distribuição do Valor da Conta por Dia da Semana')
plt.xlabel('Dia da Semana')
plt.ylabel('Valor Total da Conta ($)')
plt.show()

# Criar um violin plot para uma visão mais detalhada da distribuição
plt.figure(figsize=(12, 7))
sns.violinplot(data=tips_df, x='day', y='total_bill', order=day_order)
plt.title('Distribuição do Valor da Conta por Dia da Semana (Violin Plot)')
plt.xlabel('Dia da Semana')
plt.ylabel('Valor Total da Conta ($)')
plt.show()

Os gráficos mostram que as contas tendem a ser mais altas e mais variáveis durante o fim de semana (sábado e domingo) em comparação com os dias de semana. O violin plot, em particular, mostra a densidade dos valores, indicando que a maioria das contas aos sábados e domingos se concentra numa faixa de valores mais alta.

#### Exercício 3: Valor da Gorjeta vs. Dia da Semana

**Tarefa:** Usando o conjunto de dados `tips`, crie um box plot e um violin plot para comparar a distribuição dos valores das `tip` (gorjetas) para cada `day` da semana. Que insights pode retirar destes gráficos? Algum dia parece ter gorjetas mais altas ou maior variabilidade nas gorjetas?

## Parte 2: Análise Multivariada – Descobrindo Estruturas Complexas

A análise multivariada estende a nossa investigação para além de pares de variáveis, permitindo-nos examinar as interações entre três ou mais variáveis simultaneamente. Esta abordagem é essencial para compreender a complexidade total dos dados e para abordar desafios como a alta dimensionalidade.

### Técnicas de Visualização Avançadas

Visualizar dados com mais de três dimensões é intrinsecamente difícil. No entanto, existem técnicas inteligentes que projetam informações multidimensionais num formato 2D compreensível.

#### O Mapa de Calor de Correlação (Correlation Heatmap)

Uma extensão natural do coeficiente de correlação é a **matriz de correlação**, que calcula a correlação de Pearson para cada par de variáveis numéricas num conjunto de dados. Um **mapa de calor de correlação** visualiza esta matriz, usando a cor para representar a força e a direção da correlação. Esta é uma ferramenta poderosa para obter uma visão geral rápida de toda a teia de relações lineares, permitindo a identificação imediata de:

* **Pares de variáveis fortemente correlacionadas:** Indicado por cores intensas (próximas de +1 ou -1).
* **Potencial multicolinearidade:** Quando variáveis preditoras são altamente correlacionadas entre si, o que pode ser um problema para alguns modelos de machine learning.
* **Variáveis fortemente relacionadas com uma variável alvo.**

Vamos usar o conjunto de dados `penguins` para demonstrar.

In [ ]:
# Carregar o conjunto de dados 'penguins'
penguins_df = sns.load_dataset('penguins').dropna()

display(penguins_df.sample(5))

print(penguins_df.describe())


In [ ]:

# Selecionar apenas as colunas numéricas
numerical_penguins = penguins_df.select_dtypes(include=np.number)

# Calcular a matriz de correlação
corr_matrix = numerical_penguins.corr()

# Criar o heatmap de correlação
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Mapa de Calor de Correlação para o Conjunto de Dados Penguins')
plt.show()

O mapa de calor revela correlações fortes e positivas entre `flipper_length_mm` (comprimento da barbatana) e `body_mass_g` (massa corporal), bem como entre `culmen_length_mm` (comprimento do bico) e `flipper_length_mm`.

#### O Gráfico de Pares (Pair Plot)

O **pair plot** (ou matriz de gráficos de dispersão) é talvez a ferramenta mais abrangente para uma primeira passagem na análise exploratória multivariada. Ele cria uma matriz de gráficos onde:

* Os gráficos **fora da diagonal** são gráficos de dispersão para cada par de variáveis, mostrando as suas relações bivariadas.
* Os gráficos **na diagonal** são histogramas ou estimativas de densidade de kernel (KDE), mostrando a distribuição univariada de cada variável individual.

A verdadeira força do pair plot reside na sua capacidade de incorporar uma variável categórica através do parâmetro `hue`. Isto colore os pontos em todos os gráficos de dispersão de acordo com a categoria, revelando como os grupos se separam (ou não) no espaço de características. Usaremos o clássico conjunto de dados `iris` para ilustrar.

In [ ]:
# Carregar o conjunto de dados 'iris'
iris_df = sns.load_dataset('iris')
display(iris_df.head())
print(iris_df.describe())


In [ ]:

# Criar um pair plot, colorindo os pontos pela espécie
sns.pairplot(iris_df, hue='species', palette='viridis', diag_kind='kde')
plt.suptitle('Pair Plot do Conjunto de Dados Iris', y=1.02)
plt.show()

O pair plot do `iris` é incrivelmente informativo. Mostra claramente que a espécie `setosa` é linearmente separável das outras duas com base em quase qualquer par de características, especialmente `petal_length` e `petal_width`. As espécies `versicolor` e `virginica` são mais difíceis de separar, mas o gráfico de dispersão entre `petal_length` e `petal_width` ainda mostra uma boa distinção.

#### Exercício 4: Análise Visual do Conjunto de Dados `Iris`

**Tarefa:** Gere um pair plot para o conjunto de dados `penguin`, com os pontos coloridos por `species`. Com base no gráfico, que par de características parece ser o melhor para distinguir entre as espécies? Que espécie é mais facilmente separável das outras?

### Redução de Dimensionalidade: Simplificar para Obter Insight

Muitos conjuntos de dados modernos têm dezenas, centenas ou até milhares de características. Isto leva a um desafio conhecido como a **"maldição da dimensionalidade"**: à medida que o número de dimensões (características) aumenta, os dados tornam-se cada vez mais esparsos, e as técnicas analíticas e de modelação tornam-se computacionalmente dispendiosas e menos eficazes. A redução de dimensionalidade visa mitigar este problema, transformando os dados de um espaço de alta dimensão para um espaço de dimensão inferior, preservando o máximo de informação relevante possível.

#### Principal Component Analysis (PCA)

A Análise de Componentes Principais (PCA) é a técnica de redução de dimensionalidade mais popular. É um método **não supervisionado** que transforma os dados num novo sistema de coordenadas de "componentes principais". A ideia central é que estes novos componentes são:

1.  **Ortogonais:** Não estão correlacionados entre si.
2.  **Ordenados por variância:** O primeiro componente principal (PC1) captura a máxima variância possível nos dados. O segundo (PC2) captura a máxima variância restante, e assim por diante.

Essencialmente, o PCA encontra as direções no espaço de características ao longo das quais os dados se espalham mais.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# --- 1. Geração de Dados ---
# Cria um conjunto de dados 3D com correlação para que o PCA seja eficaz.
np.random.seed(42)
mean = [0, 0, 0]
cov = [[13, -5, 3], [-5, 5, -2], [3, -2, 2]]
data = np.random.multivariate_normal(mean, cov, 150)

# --- 2. Aplicação do PCA ---
# Centraliza os dados (passo importante para o PCA)
data_centered = data - np.mean(data, axis=0)

# Inicializa e aplica o PCA para encontrar os 3 componentes principais.
pca = PCA(n_components=3)
pca.fit(data_centered)
components = pca.components_
explained_variance = pca.explained_variance_

# Projeta os dados no novo espaço 2D dos componentes principais
data_2d = pca.transform(data_centered)[:, :2]

# --- 3. Configuração da Animação ---
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')
ax.set_box_aspect([1,1,1]) # Aspecto cúbico para melhor visualização

# Configura limites dos eixos para manter a escala consistente
max_abs = np.max(np.abs(data_centered)) * 1.1
ax.set_xlim(-max_abs, max_abs)
ax.set_ylim(-max_abs, max_abs)
ax.set_zlim(-max_abs, max_abs)


# Plota os dados centralizados
points, = ax.plot(data_centered[:, 0], data_centered[:, 1], data_centered[:, 2], 'o', color='skyblue', alpha=0.7, markersize=4)

# Linhas para os vetores dos componentes principais
arrow_pc1 = None
arrow_pc2 = None

# Título dinâmico
title = ax.set_title('Passo 1: Visualização dos Dados em 3D', fontsize=16)

# Ponto de origem (média dos dados)
mean_point = np.mean(data_centered, axis=0)

# Função de inicialização da animação
def init():
    ax.view_init(elev=20, azim=30)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    return points,

# --- 4. Função que Controla a Animação ---
def animate(i):
    global arrow_pc1, arrow_pc2
    num_frames_total = 300

    # --- FASE 1: Mostrar dados e girar (Frames 0-60) ---
    if i < 60:
        ax.view_init(elev=20, azim=30 + i * 1.5)
        title.set_text('Passo 1: Visualização dos Dados em 3D')

    # --- FASE 2: Mostrar Componentes Principais (Frames 60-120) ---
    elif i == 60:
        title.set_text('Passo 2: Encontrando Componentes Principais (PC1)')
        arrow_pc1 = ax.quiver(mean_point[0], mean_point[1], mean_point[2],
                               components[0, 0] * np.sqrt(explained_variance[0]) * 3,
                               components[0, 1] * np.sqrt(explained_variance[0]) * 3,
                               components[0, 2] * np.sqrt(explained_variance[0]) * 3,
                               color='r', linewidth=2.5, arrow_length_ratio=0.1, label='PC1')
    elif i > 60 and i < 90:
        ax.view_init(elev=20, azim=30 + 60 * 1.5)

    elif i == 90:
        title.set_text('Passo 2: Encontrando Componentes Principais (PC1 e PC2)')
        arrow_pc2 = ax.quiver(mean_point[0], mean_point[1], mean_point[2],
                               components[1, 0] * np.sqrt(explained_variance[1]) * 3,
                               components[1, 1] * np.sqrt(explained_variance[1]) * 3,
                               components[1, 2] * np.sqrt(explained_variance[1]) * 3,
                               color='g', linewidth=2.5, arrow_length_ratio=0.1, label='PC2')
    elif i > 90 and i < 120:
        ax.view_init(elev=20, azim=30 + 60 * 1.5)

    # --- FASE 3: Transição para o plano 2D alinhado (Frames 120-240) ---
    elif i >= 120 and i <= 240:
        if i == 120:
             title.set_text('Passo 3: Transição para o Plano 2D')

        # O progresso da transição vai de 0.0 a 1.0
        progress = (i - 120) / 120.0

        # Mover a câmera para uma visão de cima para baixo
        elev_inicial = 20
        azim_inicial = 30 + 60 * 1.5
        elev_final = 90  # Olhando diretamente de cima
        azim_final = -90 # Alinha o eixo X para a direita

        current_elev = elev_inicial * (1 - progress) + elev_final * progress
        current_azim = azim_inicial * (1 - progress) + azim_final * progress
        ax.view_init(elev=current_elev, azim=current_azim)

        # Onde os pontos devem estar no final: no plano XY, com
        # as coordenadas de PC1 no eixo X e PC2 no eixo Y.
        target_points = np.zeros_like(data_centered)
        target_points[:, 0] = data_2d[:, 0] # PC1 -> X
        target_points[:, 1] = data_2d[:, 1] # PC2 -> Y
        # Z é zero

        # Interpolar a posição de cada ponto do original para o alvo
        interp_data_x = data_centered[:, 0] * (1 - progress) + target_points[:, 0] * progress
        interp_data_y = data_centered[:, 1] * (1 - progress) + target_points[:, 1] * progress
        interp_data_z = data_centered[:, 2] * (1 - progress) + target_points[:, 2] * progress

        points.set_data(interp_data_x, interp_data_y)
        points.set_3d_properties(interp_data_z)

        # Esmaecer os painéis e vetores 3D no final da transição
        if i > 220:
            fade_progress = (i - 220) / 20.0
            ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 1.0 - fade_progress))
            ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 1.0 - fade_progress))
            ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 1.0 - fade_progress))
            ax.grid(False)
            if arrow_pc1: arrow_pc1.set_alpha(max(0, 1 - fade_progress))
            if arrow_pc2: arrow_pc2.set_alpha(max(0, 1 - fade_progress))

    # --- FASE 4: Visualização Final em 2D (Frames 240-300) ---
    elif i > 240:
        if i == 241:
            title.set_text('Resultado Final: Dados Reduzidos para 2D')
            ax.set_xlabel('Componente Principal 1 (PC1)', fontweight='bold')
            ax.set_ylabel('Componente Principal 2 (PC2)', fontweight='bold')
            ax.set_zticklabels([])

    return points,

# --- 5. Geração e Exibição da Animação ---
ani = FuncAnimation(fig, animate, frames=300, init_func=init, blit=False, interval=50)

plt.close(fig)
HTML(ani.to_jshtml())

#### Pré-processamento Crítico: A Necessidade do Escalamento de Características (Feature Scaling)

Um passo absolutamente crucial antes de aplicar o PCA é o **escalamento das características**. Como o PCA se baseia na maximização da variância, as características com escalas maiores (por exemplo, salários em milhares de dólares) terão naturalmente variâncias maiores do que características com escalas menores (por exemplo, anos de experiência) e, portanto, dominarão injustamente os componentes principais, mesmo que não sejam mais informativas. O escalamento, geralmente através da padronização (resultando numa média de 0 e desvio padrão de 1), garante que todas as características contribuam de forma equitativa para a análise.

##### Implementação com `scikit-learn`

Vamos aplicar o PCA ao conjunto de dados `wine`, que tem 13 características.

In [ ]:
# Carregar o conjunto de dados 'wine'
wine = load_wine(as_frame=True)

df = wine.frame
display(df.sample(5))
print(df.describe())


In [ ]:

#X = wine.data
#y = wine.target
#feature_names = wine.feature_names
X = df.drop('target', axis=1)
y = df['target']
feature_names = X.columns

# 1. Escalar os dados
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Aplicar o PCA
pca = PCA(n_components=2) # Reduzir para 2 componentes para visualização
X_pca = pca.fit_transform(X_scaled)

# Exibir a forma dos dados originais e transformados
print(f"Forma original dos dados: {X_scaled.shape}")
print(f"Forma dos dados após PCA: {X_pca.shape}")

In [ ]:
# Visualizar os resultados do PCA
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolor='k', s=50, alpha=0.8)
plt.title('PCA do Conjunto de Dados Wine (2 Componentes)')
plt.xlabel('Primeiro Componente Principal')
plt.ylabel('Segundo Componente Principal')
plt.legend(*scatter.legend_elements(), title='Classes')
plt.show()

#### Interpretação: Variância Explicada

Como sabemos se os nossos dois componentes principais capturaram informação suficiente? O atributo `explained_variance_ratio_` do objeto PCA diz-nos a percentagem da variância total do conjunto de dados que é capturada por cada componente.

In [ ]:
# Verificar a variância explicada por cada componente
print(f"Variância explicada por cada componente: {pca.explained_variance_ratio_}")
print(f"Variância total explicada pelos 2 componentes: {np.sum(pca.explained_variance_ratio_):.2f}")

Neste caso, os dois primeiros componentes principais capturam cerca de 55% da variância total. Para decidir quantos componentes reter de forma mais sistemática, podemos criar um **"scree plot"**, que é um gráfico da variância explicada acumulada em função do número de componentes. Isto ajuda a encontrar um "cotovelo" onde a adição de mais componentes proporciona ganhos marginais decrescentes.

In [ ]:
# Scree Plot para determinar o número ideal de componentes
pca_full = PCA().fit(X_scaled)
plt.figure(figsize=(10, 7))
plt.plot(np.cumsum(pca_full.explained_variance_ratio_), marker='o', linestyle='--')
plt.title('Scree Plot: Variância Explicada Acumulada por Componentes Principais')
plt.xlabel('Número de Componentes')
plt.ylabel('Variância Explicada Acumulada')
plt.axhline(y=0.9, color='r', linestyle='-')
plt.text(0.5, 0.85, '90% de variância explicada', color = 'red', fontsize=12)
plt.grid(True)
plt.show()

O scree plot mostra que precisaríamos de cerca de 6 ou 7 componentes para capturar 90% da variância nos dados do vinho.

#### Linear Discriminant Analysis (LDA)

A Análise Discriminante Linear (LDA) é uma técnica **supervisionada** utilizada para redução de dimensionalidade no contexto de um problema de classificação. A sua abordagem é fundamentalmente diferente da do PCA. Enquanto o PCA procura eixos que maximizam a variância *total* dos dados, o LDA procura eixos (discriminantes lineares) que maximizam a *separabilidade entre classes conhecidas*. O objetivo do LDA é projetar os dados num espaço de dimensão inferior de tal forma que as diferentes classes fiquem o mais distantes possível umas das outras.

A escolha entre PCA e LDA não é arbitrária; está fundamentalmente ligada ao objetivo analítico. O PCA é para explorar a estrutura inerente dos dados, enquanto o LDA é para encontrar a estrutura que melhor explica grupos pré-definidos. Se o problema é "tenho 100 características e quero ver os padrões principais", o PCA é o ponto de partida. Se o problema é "tenho 100 características e quero construir um modelo para prever a rotatividade de clientes", o LDA é um passo de pré-processamento superior, porque o seu objetivo está alinhado com a tarefa final de classificação.

A tabela seguinte contrasta estas duas técnicas fundamentais, clarificando os seus objetivos e casos de uso distintos.

| Característica | Principal Component Analysis (PCA) | Linear Discriminant Analysis (LDA) |
| :--- | :--- | :--- |
| **Tipo de Aprendizagem** | Não Supervisionado | Supervisionado |
| **Objetivo Principal** | Maximizar a variância dos dados projetados. | Maximizar a separabilidade entre as classes. |
| **Requer Rótulos?** | Não | Sim |
| **Caso de Uso** | Redução geral de dimensionalidade, EDA, redução de ruído. | Pré-processamento para classificação, maximização da distinção entre classes. |
| **Número de Componentes**| Até `n_features`. | Até `n_classes - 1`. |

##### Implementação e Comparação

Vamos aplicar tanto o PCA como o LDA ao conjunto de dados `wine` e reduzir para 2 dimensões. Depois, visualizaremos os resultados, colorindo os pontos pela classe do vinho. Esta comparação demonstrará visualmente a força do LDA em tarefas de classificação.

In [ ]:
# Aplicar LDA para reduzir para 2 componentes
lda = LinearDiscriminantAnalysis(n_components=2)
X_lda = lda.fit_transform(X_scaled, y) # LDA usa os rótulos (y)

# Criar uma figura com dois subplots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# Plot dos resultados do PCA
scatter1 = ax1.scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', edgecolor='k', s=50, alpha=0.8)
ax1.set_title('PCA do Conjunto de Dados Wine (2 Componentes)')
ax1.set_xlabel('Primeiro Componente Principal')
ax1.set_ylabel('Segundo Componente Principal')
ax1.legend(*scatter1.legend_elements(), title='Classes')

# Plot dos resultados do LDA
scatter2 = ax2.scatter(X_lda[:, 0], X_lda[:, 1], c=y, cmap='viridis', edgecolor='k', s=50, alpha=0.8)
ax2.set_title('LDA do Conjunto de Dados Wine (2 Componentes)')
ax2.set_xlabel('Primeiro Discriminante Linear')
ax2.set_ylabel('Segundo Discriminante Linear')
ax2.legend(*scatter2.legend_elements(), title='Classes')

plt.show()

A diferença é notável. Embora o PCA mostre alguma separação, os clusters sobrepõem-se consideravelmente. O LDA, por outro lado, produz uma projeção 2D com uma separação quase perfeita entre as três classes de vinho. Isto acontece porque o LDA foi explicitamente otimizado para encontrar os eixos que melhor distinguem as classes, enquanto o PCA apenas se preocupou em capturar a variância geral.

#### Exercício 5: PCA e LDA no Conjunto de Dados `Iris`

**Tarefa:** Aplique o escalamento de características e depois o PCA ao conjunto de dados `iris`, reduzindo-o a 2 componentes. Visualize o resultado num gráfico de dispersão colorido por espécie. Em seguida, aplique o LDA ao mesmo conjunto de dados, reduzindo para 2 componentes, e visualize o resultado. Compare os dois gráficos. Que técnica proporciona uma melhor separação das espécies de íris? Porque é que isto faz sentido, dados os objetivos de cada algoritmo?

### Clustering: Descobrindo Grupos Latentes

O clustering é uma tarefa de aprendizagem **não supervisionada** que visa agrupar pontos de dados de tal forma que os pontos no mesmo grupo (cluster) sejam mais semelhantes entre si do que aos de outros grupos — tudo isto sem usar quaisquer rótulos pré-existentes. É uma técnica fundamental para a segmentação de clientes, deteção de anomalias e descoberta de padrões.

#### K-Means Clustering

O K-Means é um dos algoritmos de clustering mais populares e simples. O seu funcionamento é iterativo e baseia-se em centróides:

1.  **Escolher K:** O utilizador especifica o número de clusters (K) a serem encontrados.
2.  **Inicializar Centróides:** São inicializados aleatoriamente K centróides (pontos centrais) no espaço de características.
3.  **Passo de Atribuição:** Cada ponto de dados é atribuído ao seu centróide mais próximo (geralmente usando a distância euclidiana).
4.  **Passo de Atualização:** Os centróides são recalculados como a média de todos os pontos de dados que lhes foram atribuídos.
5.  **Repetir:** Os passos 3 e 4 são repetidos até que as atribuições dos clusters não mudem mais (convergência).

In [ ]:
from sklearn.datasets import make_blobs
# --- 1. Geração de Dados ---
# Gera dados sintéticos com agrupamentos claros para facilitar a visualização.
n_samples = 300
n_features = 2
n_clusters = 6
random_state = 50

X, y = make_blobs(n_samples=n_samples,
                  n_features=n_features,
                  centers=n_clusters,
                  random_state=random_state)

# --- 2. Inicialização dos Centros ---
# Escolhe 'k' pontos de dados aleatórios para serem os centróides iniciais.
np.random.seed(random_state)
random_indices = np.random.choice(n_samples, size=n_clusters, replace=False)
centroids = X[random_indices, :]

# --- 3. Configuração da Animação ---
fig, ax = plt.subplots(figsize=(8, 6))
plt.style.use('seaborn-v0_8-whitegrid')

# Função que será chamada a cada quadro da animação
def update(frame):
    global centroids # Usa os centróides da iteração anterior

    # Limpa o gráfico anterior para desenhar o novo quadro
    ax.clear()

    # --- Passo de Atribuição (Assignment Step) ---
    # Calcula a distância de cada ponto a cada centróide e atribui ao mais próximo.
    distances = np.sqrt(((X - centroids[:, np.newaxis])**2).sum(axis=2))
    closest_centroid = np.argmin(distances, axis=0)

    # Guarda a posição antiga dos centróides para desenhar as setas de movimento
    old_centroids = centroids.copy()

    # --- Passo de Atualização (Update Step) ---
    # Recalcula a posição dos centróides como a média dos pontos atribuídos a eles.
    for i in range(n_clusters):
        points_in_cluster = X[closest_centroid == i]
        if len(points_in_cluster) > 0:
            centroids[i] = np.mean(points_in_cluster, axis=0)

    # --- Desenho do Quadro Atual ---
    # Plota os pontos de dados, coloridos de acordo com o centróide atribuído.
    scatter = ax.scatter(X[:, 0], X[:, 1], c=closest_centroid, cmap='viridis', alpha=0.6)

    # Plota os centróides da iteração atual como estrelas maiores
    ax.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='*', s=300, edgecolor='black', label='Novos Centróides')

    # Plota os centróides da iteração anterior e desenha setas para mostrar o movimento
    ax.scatter(old_centroids[:, 0], old_centroids[:, 1], c='black', marker='x', s=100, label='Centróides Antigos')
    for i in range(n_clusters):
        ax.annotate('', xy=centroids[i], xytext=old_centroids[i],
                    arrowprops=dict(arrowstyle="->", color='red', lw=1.5))

    # --- Configurações Visuais do Gráfico ---
    ax.set_title(f'K-Means: Iteração {frame + 1}', fontsize=16)
    ax.set_xlabel('Característica 1')
    ax.set_ylabel('Característica 2')
    ax.legend(loc='upper right')
    ax.grid(True, which='both', linestyle='--', linewidth=0.5)

# --- 4. Criação e Exibição da Animação ---
# Cria a animação chamando a função 'update' 10 vezes (10 iterações)
# O intervalo é de 1500 ms (1.5 segundos) entre os quadros.
ani = FuncAnimation(fig, update, frames=10, interval=1500, repeat=False)

# Converte a animação para HTML para ser exibida no notebook
html_animation = HTML(ani.to_jshtml())

# Fecha o plot estático para evitar duplicatas
plt.close()

# Exibe a animação
html_animation


##### Uma Decisão Crítica: Usar o Método do Cotovelo (Elbow Method) para Escolher K

O maior desafio no K-Means é a escolha do valor de K. O **Método do Cotovelo** é uma heurística popular para ajudar nesta decisão. Baseia-se no conceito de **inércia**, também conhecido como Soma dos Quadrados Dentro do Cluster (WCSS - Within-Cluster Sum of Squares). A inércia mede quão internamente coerentes são os clusters; valores mais baixos são melhores.

$$\text{WCSS} = \sum_ {k=1}^ {K} \sum_ {i=1}^ {n_k} (x_i – c_k)^2$$

O método funciona da seguinte forma:
1.  Executar o K-Means para um intervalo de valores de K (por exemplo, de 1 a 10).
2.  Para cada K, calcular a inércia total.
3.  Traçar um gráfico de K vs. Inércia.
4.  Procurar o ponto "cotovelo" no gráfico — o ponto onde a taxa de diminuição da inércia abranda significativamente. Este ponto sugere um bom equilíbrio entre o número de clusters e a coesão dentro dos clusters, indicando que adicionar mais clusters proporciona retornos decrescentes.

##### Implementação com `scikit-learn`

Vamos usar o conjunto de dados `penguins` para segmentar os pinguins com base nas suas características numéricas. Como o K-Means é um algoritmo baseado em distância, o escalamento das características é, mais uma vez, um passo de pré-processamento importante.

In [ ]:
penguins_df.columns

In [ ]:
# Usar as características numéricas do conjunto de dados penguins
X_penguins = penguins_df[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']]

# Escalar os dados
scaler_penguins = StandardScaler()
X_penguins_scaled = scaler_penguins.fit_transform(X_penguins)

# Implementar o Método do Cotovelo
inertia = []
K_range = range(1, 11)
for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_penguins_scaled)
    inertia.append(kmeans.inertia_)

# Plotar o gráfico do cotovelo
plt.figure(figsize=(10, 7))
plt.plot(K_range, inertia, marker='o', linestyle='--')
plt.title('Método do Cotovelo para Escolha Ótima de K')
plt.xlabel('Número de Clusters (K)')
plt.ylabel('Inércia (WCSS)')
plt.xticks(K_range)
plt.grid(True)
plt.show()

O gráfico do cotovelo mostra uma clara inflexão em K=3. A inércia diminui acentuadamente de K=1 para K=3, mas a diminuição torna-se muito mais gradual depois disso. Isto sugere que 3 é o número ideal de clusters para estes dados.

### Visualização

Agora, vamos executar o K-Means com K=3 e visualizar os clusters resultantes. Para a visualização, usaremos as duas primeiras características (`culmen_length_mm` e `culmen_depth_mm`) e coloriremos os pontos de acordo com o rótulo do cluster atribuído. Também marcaremos os centróides finais.

In [ ]:
# Executar K-Means com o K ótimo (K=3)
optimal_k = 3
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_penguins_scaled)

# Adicionar os rótulos dos clusters ao dataframe original para facilitar a plotagem
penguins_df['cluster'] = cluster_labels

# Visualizar os clusters
plt.figure(figsize=(12, 8))
sns.scatterplot(data=penguins_df, x='bill_length_mm', y='bill_depth_mm', hue='cluster', palette='viridis', s=100, alpha=0.8)

# Plotar os centróides
# (Nota: os centróides estão no espaço escalado, precisamos de os transformar de volta para a escala original para plotar)
centroids_scaled = kmeans.cluster_centers_
# Precisamos criar um scaler que foi treinado apenas nas colunas usadas para o plot
scaler_plot = StandardScaler()
scaler_plot.fit(X_penguins[['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']])
centroids_original_scale = scaler_penguins.inverse_transform(centroids_scaled)

plt.scatter(centroids_original_scale[:, 0], centroids_original_scale[:, 1],
            s=300, c='red', marker='X', label='Centróides')

plt.title('Clusters de Pinguins Encontrados pelo K-Means (K=3)')
plt.xlabel('Comprimento do Bico (mm)')
plt.ylabel('Profundidade do Bico (mm)')
plt.legend()
plt.show()

O K-Means identificou com sucesso três grupos distintos nos dados, que, como seria de esperar, correspondem de perto às três espécies de pinguins no conjunto de dados.

#### Exercício 6: Segmentando o Conjunto de Dados `Wine`

**Tarefa:** Realize o clustering usando K-Means na base Wine. Escolha apenas algumas dimensões se quiser. Primeiro, use o Método do Cotovelo para determinar o número ideal de clusters. Em seguida, execute o K-Means com o K ótimo e crie um gráfico de dispersão mostrando os pontos de dados coloridos pelo seu cluster atribuído e os centróides finais do cluster. Quão bem os clusters que encontrou correspondem às espécies reais de pinguins no conjunto de dados? (Pode verificar criando um gráfico semelhante, mas com `hue='target'`).

## Conclusão: Sintetizando o Seu Kit de Ferramentas Analíticas

Nesta aula, viajámos desde a análise de simples pares de variáveis até à exploração de estruturas multidimensionais complexas. Cobrimos um conjunto de técnicas essenciais que formam o núcleo da análise exploratória de dados moderna:

* **Análise de Correlação e Gráficos de Dispersão** para quantificar e visualizar relações lineares entre variáveis numéricas.
* **Tabelas de Contingência e Mapas de Calor** para descobrir associações entre variáveis categóricas.
* **Box Plots e Violin Plots** para comparar distribuições de variáveis numéricas entre diferentes grupos.
* **Mapas de Calor de Correlação e Pair Plots** para obter uma visão geral abrangente das interações multivariadas.
* **Principal Component Analysis (PCA)** como uma ferramenta não supervisionada para reduzir a dimensionalidade, maximizando a variância e simplificando dados complexos.
* **Linear Discriminant Analysis (LDA)** como uma técnica supervisionada para reduzir a dimensionalidade, maximizando a separabilidade entre classes para tarefas de classificação.
* **K-Means Clustering** para descobrir grupos latentes e segmentar dados com base na similaridade, sem a necessidade de rótulos prévios.

É crucial entender que estas não são ferramentas isoladas, mas sim um kit de ferramentas integrado. A análise de correlação informa o que procurar num pair plot. A visualização de dados com PCA ou LDA pode revelar a necessidade de clustering. E, mais importante, os resultados destas análises — características de dimensão reduzida do PCA/LDA ou rótulos de cluster do K-Means — frequentemente servem como entradas valiosas para a próxima fase do fluxo de trabalho da ciência de dados: a modelação preditiva. Ao dominar estas técnicas, está equipado para transformar dados brutos e complexos em insights estruturados e acionáveis.